# Speech Emotion Recognition on RAVDESS: Model 2, ResNet-18 transfer learning

This notebook reuses **the same RAVDESS preprocessing, speaker-independent split, normalisation, augmentation, metrics, and evaluation protocol as Model 1**. The only major change is the classifier: a **ResNet-18 pretrained on ImageNet** is used as a frozen feature extractor, the single-channel spectrogram is copied to three channels, and only a new fully connected emotion head is trained.

This matches the course Model 2 idea: **pretrained ResNet-18 + 3-channel spectrogram + frozen backbone + new trained head**.

## 1. Setup: imports, seeds, device, hyperparameters

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS

import importlib
import json
import platform
import random
import re
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import sklearn
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_fscore_support, recall_score)
from IPython.display import Audio, display
from tqdm.auto import tqdm

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

print(f"Python {platform.python_version()} | NumPy {np.__version__} | pandas {pd.__version__} | "
      f"librosa {librosa.__version__} | torch {torch.__version__} | scikit-learn {sklearn.__version__}")

In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
SEED = 42


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


seed_everything(SEED)

# ---------------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"Device: cuda ({torch.cuda.get_device_name(0)})")
else:
    print("Device: cpu  (no GPU detected: training will be slow; enable Settings -> Accelerator -> GPU)")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
MODEL_DIR = WORKING_ROOT / "model2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = MODEL_DIR / "best_model.pt"

# ---------------------------------------------------------------------------
# Speaker split: RAVDESS actors, odd = male, even = female
# ---------------------------------------------------------------------------
SPLIT_SEED = 42
N_TEST_PER_SEX = 2          # 4 test actors
N_VAL_PER_SEX = 2           # 4 validation actors  -> 16 training actors

# ---------------------------------------------------------------------------
# Training hyperparameters
# Keep the same optimisation/evaluation setup as Model 1 for a fair comparison.
# Only the new ResNet classification head is trainable.
# ---------------------------------------------------------------------------
HPARAMS = {
    "batch_size": 32,
    "max_epochs": 60,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "dropout": 0.3,
    "class_weighted_loss": True,       # neutral has half as many clips as the other emotions
    "lr_scheduler": "ReduceLROnPlateau(val UAR, factor=0.5, patience=4, min_lr=1e-5)",
    "early_stopping_patience": 12,     # epochs without validation-UAR improvement
    # SpecAugment-style augmentation (training batches only)
    "max_time_shift_frames": 30,       # +/- 0.30 s
    "n_time_masks": 2, "max_time_mask_frames": 20,     # up to 0.20 s each
    "n_freq_masks": 2, "max_freq_mask_bands": 8,       # up to 8 of 64 Mel bands each
}
print(json.dumps(HPARAMS, indent=2))

## 2. Shared module: preprocessing, normalisation, ResNet-18 and inference

The next cell writes `ser_model2.py` into `/kaggle/working/model2/`. The audio preprocessing and training-set normalisation stay identical to Model 1 so the comparison is fair. The model itself changes to a pretrained ResNet-18: the spectrogram is copied from one channel to three, the ImageNet backbone is frozen, and only a new emotion-classification head is trainable.


In [ ]:
%%writefile /kaggle/working/model2/ser_model2.py
"""Model 2 toolkit shared by the training notebook and the web backend.

    recorded voice -> extract_log_mel -> Normalizer.apply -> EmotionResNet18 -> emotion

The training notebook imports this file, and the same file ships with the saved
model, so training and live inference use identical code.
"""
from __future__ import annotations

import io
import json
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
from torchvision.models import ResNet18_Weights, resnet18

# =============================================================================
# Classes (index order is part of the model: never reorder after training)
# =============================================================================
EMOTIONS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]

# =============================================================================
# Preprocessing configuration (identical to the validated preprocessing notebook)
# =============================================================================
SAMPLE_RATE = 16_000
DURATION_S = 3.0
NUM_SAMPLES = 48_000
WIN_LENGTH = 400          # 25 ms
HOP_LENGTH = 160          # 10 ms
N_FFT = 512
N_MELS = 64
FMIN, FMAX = 0.0, 8000.0
DB_REF, AMIN = 1.0, 1e-10
DB_FLOOR = -100.0
EXPECTED_SHAPE = (64, 301)

PREPROCESSING_CONFIG = {
    "sample_rate": SAMPLE_RATE, "duration_s": DURATION_S, "num_samples": NUM_SAMPLES,
    "crop": "center", "pad": "center (zeros)",
    "win_length": WIN_LENGTH, "hop_length": HOP_LENGTH, "n_fft": N_FFT,
    "window": "hann", "center": True, "stft_pad_mode": "constant",
    "n_mels": N_MELS, "fmin": FMIN, "fmax": FMAX, "mel_scale": "slaney", "mel_norm": "slaney",
    "power": 2.0, "db_ref": DB_REF, "amin": AMIN, "db_floor": DB_FLOOR, "top_db": None,
    "resampler": "soxr_hq", "dtype": "float32", "expected_shape": list(EXPECTED_SHAPE),
}


# =============================================================================
# Audio -> log-Mel
# =============================================================================
def load_audio(source, sr=None):
    """File path, WAV bytes or NumPy array (+ sr) -> (waveform, sample_rate)."""
    if isinstance(source, (str, Path)):
        y, native_sr = sf.read(str(source), dtype="float32", always_2d=True)
    elif isinstance(source, (bytes, bytearray)):
        y, native_sr = sf.read(io.BytesIO(bytes(source)), dtype="float32", always_2d=True)
    elif isinstance(source, np.ndarray):
        if sr is None:
            raise ValueError("sr is required for a NumPy waveform")
        y, native_sr = np.asarray(source, dtype=np.float32), sr
        y = y if y.ndim == 1 else y.T            # (channels, samples) -> (samples, channels)
        y = y[:, None] if y.ndim == 1 else y
    else:
        raise TypeError(f"unsupported audio source: {type(source).__name__}")
    if y.size == 0 or not np.all(np.isfinite(y)):
        raise ValueError("audio is empty or contains NaN/inf")
    return y.T, int(native_sr)                    # (channels, samples)


def standardize_audio(y, sr):
    """(channels, samples) at any rate -> mono, 16 kHz, exactly 3 s (centre crop / pad)."""
    mono = y.mean(axis=0)
    if sr != SAMPLE_RATE:
        mono = librosa.resample(mono, orig_sr=sr, target_sr=SAMPLE_RATE, res_type="soxr_hq")
    n = mono.shape[0]
    if n > NUM_SAMPLES:
        start = (n - NUM_SAMPLES) // 2
        mono = mono[start:start + NUM_SAMPLES]
    elif n < NUM_SAMPLES:
        left = (NUM_SAMPLES - n) // 2
        mono = np.pad(mono, (left, NUM_SAMPLES - n - left))
    return np.ascontiguousarray(mono, dtype=np.float32)


def waveform_to_log_mel(waveform):
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        window="hann", center=True, pad_mode="constant", power=2.0,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, htk=False, norm="slaney",
    )
    log_mel = librosa.power_to_db(mel, ref=DB_REF, amin=AMIN, top_db=None).astype(np.float32)
    if log_mel.shape != EXPECTED_SHAPE or not np.all(np.isfinite(log_mel)):
        raise ValueError(f"invalid log-Mel {log_mel.shape}")
    return log_mel


def extract_log_mel(source, sr=None):
    """Central preprocessing function: any audio source -> (64, 301) float32 dB matrix."""
    y, native_sr = load_audio(source, sr)
    return waveform_to_log_mel(standardize_audio(y, native_sr))


# =============================================================================
# Normalisation (statistics computed on TRAINING speakers only)
# =============================================================================
class Normalizer:
    """Per-Mel-band standardisation: (x - mean[band]) / std[band]."""

    def __init__(self, mean, std, eps=1e-6, source="training speakers"):
        self.mean = np.asarray(mean, dtype=np.float32).reshape(-1)
        self.std = np.asarray(std, dtype=np.float32).reshape(-1)
        self.eps = float(eps)
        self.source = source

    @classmethod
    def fit(cls, features, source="training speakers"):
        """features: array (N, n_mels, frames) from the training set only."""
        x = np.asarray(features, dtype=np.float64)
        return cls(x.mean(axis=(0, 2)), x.std(axis=(0, 2)), source=source)

    def apply(self, log_mel):
        """Works on (n_mels, frames) or (N, n_mels, frames)."""
        return ((log_mel - self.mean[:, None]) / (self.std[:, None] + self.eps)).astype(np.float32)

    def to_dict(self):
        return {"mode": "per_mel_band", "source": self.source, "eps": self.eps,
                "mean": self.mean.tolist(), "std": self.std.tolist()}

    def save(self, path):
        Path(path).write_text(json.dumps(self.to_dict(), indent=2))

    @classmethod
    def load(cls, path):
        d = json.loads(Path(path).read_text())
        return cls(d["mean"], d["std"], d["eps"], d.get("source", ""))


# =============================================================================
# Model 2: ResNet-18 transfer learning
# =============================================================================
class EmotionResNet18(nn.Module):
    """Frozen ImageNet ResNet-18 backbone + trainable emotion classifier.

    Input from the dataset remains (B, 1, 64, 301). In forward(), the same
    spectrogram is copied into three channels before entering ResNet-18.
    """

    def __init__(self, n_classes=8, pretrained=True):
        super().__init__()
        self.config = {"n_classes": int(n_classes)}

        weights = ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = resnet18(weights=weights)
        feature_dim = self.backbone.fc.in_features       # 512 for ResNet-18
        self.backbone.fc = nn.Identity()

        # Course Model 2: freeze the whole pretrained backbone.
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

        # Only this new head is trained on RAVDESS.
        self.classifier = nn.Linear(feature_dim, n_classes)

    def train(self, mode=True):
        """Train the new head while keeping the frozen backbone/BatchNorm in eval mode."""
        super().train(mode)
        self.backbone.eval()
        return self

    def forward(self, x):
        # Spectrograms are grayscale-like: (B, 1, 64, 301).
        # ResNet-18 expects 3 input channels, so copy the same spectrogram 3 times.
        if x.ndim != 4:
            raise ValueError(f"expected a 4-D tensor (B,C,H,W), got {tuple(x.shape)}")
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        elif x.shape[1] != 3:
            raise ValueError(f"expected 1 or 3 input channels, got {x.shape[1]}")

        features = self.backbone(x)                      # (B, 512)
        return self.classifier(features)                 # (B, n_classes)


# =============================================================================
# Inference (what the web backend calls)
# =============================================================================
def load_model(model_dir, device="cpu"):
    """Load saved Model 2 weights, class order and training-set normalisation."""
    model_dir = Path(model_dir)
    checkpoint = torch.load(model_dir / "best_model.pt", map_location=device, weights_only=True)

    # No ImageNet download is needed at inference: the checkpoint already contains
    # the complete pretrained backbone weights plus the trained classification head.
    model = EmotionResNet18(**checkpoint["model_config"], pretrained=False).to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    normalizer = Normalizer.load(model_dir / "normalization_stats.json")
    classes = checkpoint["classes"]
    return model, normalizer, classes


@torch.no_grad()
def predict(source, model, normalizer, classes, sr=None, device="cpu"):
    """Audio -> log-Mel -> train-set normalisation -> 3-channel ResNet-18 -> probabilities."""
    log_mel = extract_log_mel(source, sr=sr)
    x = torch.from_numpy(normalizer.apply(log_mel))[None, None].to(device)  # (1,1,64,301)
    probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
    best = int(probs.argmax())
    return {
        "emotion": classes[best],
        "confidence": float(probs[best]),
        "probabilities": {c: float(p) for c, p in zip(classes, probs)},
        "log_mel": log_mel,
    }


In [ ]:
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))
import ser_model2
ser_model2 = importlib.reload(ser_model2)      # picks up edits if the cell above is re-run

from ser_model2 import (EMOTIONS, EXPECTED_SHAPE, DB_FLOOR, PREPROCESSING_CONFIG,
                        EmotionResNet18, Normalizer, extract_log_mel)

CLASS_TO_INDEX = {emotion: i for i, emotion in enumerate(EMOTIONS)}
INDEX_TO_CLASS = {i: emotion for emotion, i in CLASS_TO_INDEX.items()}
N_CLASSES = len(EMOTIONS)
INPUT_SHAPE = (1, *EXPECTED_SHAPE)             # stored spectrogram shape before 3-channel copy
RESNET_INPUT_SHAPE = (3, *EXPECTED_SHAPE)      # actual shape entering ResNet-18
print(f"Classes ({N_CLASSES}): {CLASS_TO_INDEX}")
print(f"Stored spectrogram shape: {INPUT_SHAPE}")
print(f"ResNet input after channel copy: {RESNET_INPUT_SHAPE}")

## 3. Locate RAVDESS and decode the metadata

WAV files are found recursively under `/kaggle/input`. The Kaggle copy of RAVDESS contains every recording twice
(`Actor_XX/` and `audio_speech_actors_01-24/Actor_XX/`), so one copy per file name is kept, as in the preprocessing notebook.

Speaker and emotion come from the file name `MM-VV-EE-II-SS-RR-AA.wav` (field 3 = emotion, field 7 = actor).

In [ ]:
RAVDESS_PATTERN = re.compile(r"^(\d{2})-(\d{2})-(\d{2})-(\d{2})-(\d{2})-(\d{2})-(\d{2})$")
EMOTION_CODES = {"01": "neutral", "02": "calm", "03": "happy", "04": "sad",
                 "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised"}


def parse_ravdess_filename(path):
    match = RAVDESS_PATTERN.match(Path(path).stem)
    if match is None:
        raise ValueError(f"{Path(path).name}: not a RAVDESS file name")
    modality, channel, emotion, intensity, statement, repetition, actor = match.groups()
    if emotion not in EMOTION_CODES:
        raise ValueError(f"{Path(path).name}: unknown emotion code {emotion}")
    return {
        "filename": Path(path).name,
        "speaker_id": actor,
        "speaker_key": f"ravdess/{actor}",
        "sex": "male" if int(actor) % 2 == 1 else "female",
        "emotion": EMOTION_CODES[emotion],
        "label": CLASS_TO_INDEX[EMOTION_CODES[emotion]],
        "intensity": {"01": "normal", "02": "strong"}.get(intensity, intensity),
        "statement": statement,
        "repetition": repetition,
        "vocal_channel": {"01": "speech", "02": "song"}.get(channel, channel),
    }


all_wavs = sorted(p for p in INPUT_ROOT.rglob("*") if p.is_file() and p.suffix.lower() == ".wav")
if not all_wavs:
    raise FileNotFoundError("No .wav files under /kaggle/input. Attach 'uwrfkaggler/ravdess-emotional-speech-audio' "
                            "(Add Input) and Run All again.")

by_name = defaultdict(list)
for p in all_wavs:
    by_name[p.name].append(p)
unique_wavs = [sorted(paths, key=lambda p: (len(p.parts), str(p)))[0] for _, paths in sorted(by_name.items())]

rows, skipped = [], []
for p in unique_wavs:
    try:
        rows.append({**parse_ravdess_filename(p), "wav_path": str(p)})
    except ValueError as exc:
        skipped.append(str(exc))

data = pd.DataFrame(rows)
data = data[data["vocal_channel"] == "speech"].sort_values(["speaker_key", "filename"]).reset_index(drop=True)

print(f"WAV files found: {len(all_wavs)} | unique recordings: {len(unique_wavs)} | "
      f"usable speech recordings: {len(data)} | skipped names: {len(skipped)}")
print(f"Speakers: {data['speaker_key'].nunique()} | emotions: {sorted(data['emotion'].unique(), key=EMOTIONS.index)}")
missing_classes = set(EMOTIONS) - set(data["emotion"])
assert not missing_classes, f"classes absent from the data: {missing_classes}"

## 4. Load the log-Mel features

To avoid recomputing, the notebook first looks for the output of the preprocessing notebook (`manifest.csv` +
`config.json` + `features/`), either in `/kaggle/working/ravdess_features/` or attached as an input dataset.
It is used only if its `config.json` matches the preprocessing configuration **exactly**, and a few files are
recomputed with `extract_log_mel` to confirm that the cached features match the current code.

Otherwise, all features are computed with `extract_log_mel` (about 1 to 2 minutes).
Everything is held in memory: $1440 \times 64 \times 301 \times 4$ bytes $\approx 111$ MB.

In [ ]:
def find_feature_cache():
    candidates = [WORKING_ROOT / "ravdess_features"] + [p.parent for p in INPUT_ROOT.rglob("manifest.csv")]
    expected = json.loads(json.dumps(PREPROCESSING_CONFIG))
    for root in candidates:
        config_path, manifest_path = root / "config.json", root / "manifest.csv"
        if config_path.exists() and manifest_path.exists():
            if json.loads(config_path.read_text()) == expected:
                return root
            print(f"Ignoring {root}: its config.json differs from the current preprocessing configuration.")
    return None


def load_features(frame, cache_root):
    cached = {}
    if cache_root is not None:
        manifest = pd.read_csv(cache_root / "manifest.csv", dtype=str)
        cached = {r.filename: cache_root / r.feature_relpath for r in manifest.itertuples()}
    features = np.empty((len(frame), *EXPECTED_SHAPE), dtype=np.float32)
    n_cached = 0
    for i, row in enumerate(tqdm(frame.itertuples(), total=len(frame), desc="Loading log-Mel", mininterval=1.0)):
        path = cached.get(row.filename)
        if path is not None and path.exists():
            features[i] = np.load(path, allow_pickle=False)
            n_cached += 1
        else:
            features[i] = extract_log_mel(row.wav_path)
    return features, n_cached


feature_cache = find_feature_cache()
print(f"Feature cache: {feature_cache if feature_cache else 'none found, computing from WAV files'}")
t0 = time.time()
X_db, n_from_cache = load_features(data, feature_cache)
print(f"Loaded {len(X_db)} matrices ({n_from_cache} from cache, {len(X_db) - n_from_cache} computed) "
      f"in {time.time() - t0:.1f} s | array {X_db.shape} {X_db.dtype}")

if n_from_cache:
    check_idx = np.random.default_rng(SEED).choice(len(data), size=min(3, len(data)), replace=False)
    max_diff = max(float(np.abs(extract_log_mel(data.wav_path[i]) - X_db[i]).max()) for i in check_idx)
    print(f"Cache consistency (max |cached - recomputed| on {len(check_idx)} files): {max_diff:.2e} dB")
    if max_diff > 1e-2:
        print("Cache does not match the current code: recomputing every feature.")
        X_db, _ = load_features(data, None)

assert X_db.shape[1:] == EXPECTED_SHAPE and X_db.dtype == np.float32 and np.isfinite(X_db).all()
y_all = data["label"].to_numpy(dtype=np.int64)

## 5. Speaker-independent split

Recordings are **never** split at random: the unit of the split is the actor.
The 24 actors are split **16 / 4 / 4**, with 2 male and 2 female actors in validation and 2 + 2 in test, so each
set covers both sexes. The split is drawn once with a fixed seed and saved.

A test set of 4 actors (240 clips) is small, so test scores carry noticeable uncertainty; the class slides mention
leave-speakers-out cross-validation as the more robust alternative for later.

In [ ]:
actors = sorted(data["speaker_id"].unique())
male_actors = [a for a in actors if int(a) % 2 == 1]
female_actors = [a for a in actors if int(a) % 2 == 0]
split_rng = np.random.default_rng(SPLIT_SEED)
male_order = list(split_rng.permutation(male_actors))
female_order = list(split_rng.permutation(female_actors))

test_speakers = sorted(male_order[:N_TEST_PER_SEX] + female_order[:N_TEST_PER_SEX])
val_speakers = sorted(male_order[N_TEST_PER_SEX:N_TEST_PER_SEX + N_VAL_PER_SEX]
                      + female_order[N_TEST_PER_SEX:N_TEST_PER_SEX + N_VAL_PER_SEX])
train_speakers = sorted(set(actors) - set(test_speakers) - set(val_speakers))
SPLIT = {"train": train_speakers, "validation": val_speakers, "test": test_speakers}

data["split"] = data["speaker_id"].map({s: name for name, ids in SPLIT.items() for s in ids})
idx = {name: np.flatnonzero(data["split"].to_numpy() == name) for name in SPLIT}

# --- Proof of speaker independence --------------------------------------------
speakers_in = {name: set(data.loc[idx[name], "speaker_id"]) for name in SPLIT}
assert not speakers_in["train"] & speakers_in["validation"], "train/validation speaker overlap"
assert not speakers_in["train"] & speakers_in["test"], "train/test speaker overlap"
assert not speakers_in["validation"] & speakers_in["test"], "validation/test speaker overlap"
assert sum(len(v) for v in idx.values()) == len(data), "some recordings are not assigned"
print("Speaker overlap between splits: none (assertions passed)\n")

for name in SPLIT:
    print(f"{name:<10} speakers ({len(SPLIT[name]):>2}): {', '.join(SPLIT[name])}  -> {len(idx[name])} recordings")

distribution = pd.crosstab(data["split"], data["emotion"]).reindex(index=list(SPLIT), columns=EMOTIONS)
distribution["total"] = distribution.sum(axis=1)
print("\nClass distribution per split:")
display(distribution)

(MODEL_DIR / "speaker_split.json").write_text(json.dumps({
    "unit": "speaker (RAVDESS actor)", "seed": SPLIT_SEED,
    "train": train_speakers, "validation": val_speakers, "test": test_speakers,
    "n_recordings": {name: int(len(idx[name])) for name in SPLIT},
}, indent=2))
data.to_csv(MODEL_DIR / "split_manifest.csv", index=False)

## 6. Normalisation fitted on training speakers only

For each Mel band $b$, the mean $\mu_b$ and standard deviation $\sigma_b$ are computed over all training frames, then

$$
\tilde{x}_{b,t} = \frac{x_{b,t} - \mu_b}{\sigma_b + 10^{-6}} .
$$

The **same** $\mu, \sigma$ (from training speakers) are applied to train, validation, test and, later, live web
recordings. Validation and test data never influence them. The statistics are saved to `normalization_stats.json`.

In [ ]:
normalizer = Normalizer.fit(X_db[idx["train"]], source=f"training speakers {', '.join(train_speakers)}")
normalizer.save(MODEL_DIR / "normalization_stats.json")

X = normalizer.apply(X_db)        # train-derived statistics applied to every split

print(f"Mean per band (dB): {normalizer.mean.min():.1f} to {normalizer.mean.max():.1f}")
print(f"Std per band  (dB): {normalizer.std.min():.1f} to {normalizer.std.max():.1f}")
for name in SPLIT:
    part = X[idx[name]]
    print(f"After normalisation, {name:<10}: mean {part.mean():+.3f}, std {part.std():.3f}")
print("(train is 0 / 1 by construction; validation and test are close but not exact, as expected)")

## 7. Datasets and training-time augmentation

Augmentation is applied **only to training batches**, on the normalised matrices, and kept moderate:

| Transform | Setting | Why it preserves emotion |
|---|---|---|
| Random time shift | up to ±30 frames (0.3 s), vacated frames filled with silence | emotion does not depend on exact onset time |
| Time masking | up to 2 masks of ≤ 20 frames | hides short segments; prosody remains |
| Frequency masking | up to 2 masks of ≤ 8 of 64 bands | forces use of several frequency regions |

Masked values are set to 0, which after normalisation is the training mean of that band.
Randomness comes from a seeded `torch.Generator`, so runs are reproducible.

In [ ]:
class SpectrogramDataset(Dataset):
    def __init__(self, features, labels, indices):
        self.x = torch.from_numpy(features[indices])
        self.y = torch.from_numpy(labels[indices])
        self.indices = torch.from_numpy(indices)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.x[i].unsqueeze(0), self.y[i], self.indices[i]     # (1, 64, 301)


# Normalised value of digital silence (-100 dB) in each band, used to fill shifted-in frames.
SILENCE_FILL = torch.from_numpy(normalizer.apply(np.full((EXPECTED_SHAPE[0], 1), DB_FLOOR, np.float32))[:, 0])


def _randint(low, high, generator):
    return int(torch.randint(low, high + 1, (1,), generator=generator))


def augment_batch(x, generator, hp=HPARAMS):
    # x: (B, 1, n_mels, frames), already normalised. Returns an augmented copy.
    out = x.clone()
    _, _, n_mels, n_frames = out.shape
    fill = SILENCE_FILL.to(out.device)[:, None]
    for i in range(out.shape[0]):
        shift = _randint(-hp["max_time_shift_frames"], hp["max_time_shift_frames"], generator)
        if shift > 0:
            out[i, 0, :, shift:] = x[i, 0, :, :-shift]
            out[i, 0, :, :shift] = fill
        elif shift < 0:
            out[i, 0, :, :shift] = x[i, 0, :, -shift:]
            out[i, 0, :, shift:] = fill
        for _ in range(_randint(0, hp["n_time_masks"], generator)):
            width = _randint(0, hp["max_time_mask_frames"], generator)
            start = _randint(0, n_frames - width, generator)
            out[i, 0, :, start:start + width] = 0.0
        for _ in range(_randint(0, hp["n_freq_masks"], generator)):
            width = _randint(0, hp["max_freq_mask_bands"], generator)
            start = _randint(0, n_mels - width, generator)
            out[i, 0, start:start + width, :] = 0.0
    return out


datasets = {name: SpectrogramDataset(X, y_all, idx[name]) for name in SPLIT}
loader_generator = torch.Generator().manual_seed(SEED)
augment_generator = torch.Generator().manual_seed(SEED + 1)
loaders = {
    "train": DataLoader(datasets["train"], batch_size=HPARAMS["batch_size"], shuffle=True,
                        generator=loader_generator, num_workers=0, drop_last=False),
    "validation": DataLoader(datasets["validation"], batch_size=128, shuffle=False, num_workers=0),
    "test": DataLoader(datasets["test"], batch_size=128, shuffle=False, num_workers=0),
}
print({name: f"{len(ds)} clips / {len(loaders[name])} batches" for name, ds in datasets.items()})

In [ ]:
def plot_spectrogram(matrix, ax, title, colorbar_label="dB", cmap="magma", **kwargs):
    # Human view of a (64, 301) matrix: time on x, Mel frequency on y.
    mesh = librosa.display.specshow(matrix, sr=ser_model2.SAMPLE_RATE, hop_length=ser_model2.HOP_LENGTH,
                                    n_fft=ser_model2.N_FFT, x_axis="time", y_axis="mel",
                                    fmin=ser_model2.FMIN, fmax=ser_model2.FMAX, cmap=cmap, ax=ax, **kwargs)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Mel frequency (Hz)")
    plt.colorbar(mesh, ax=ax, label=colorbar_label)
    return mesh


# One training example before and after augmentation (normalised units).
example_x, example_y, _ = datasets["train"][0]
preview_generator = torch.Generator().manual_seed(SEED + 2)
example_aug = augment_batch(example_x[None], preview_generator)[0]

fig, axes = plt.subplots(1, 2, figsize=(15, 3.6), layout="constrained")
plot_spectrogram(example_x[0].numpy(), axes[0], f"Normalised input ({INDEX_TO_CLASS[int(example_y)]})",
                 colorbar_label="std units", vmin=-3, vmax=3)
plot_spectrogram(example_aug[0].numpy(), axes[1], "Same input after augmentation (training only)",
                 colorbar_label="std units", vmin=-3, vmax=3)
plt.show()

## 8. Model 2: ResNet-18 transfer learning

The original ImageNet classifier is removed. The ResNet-18 backbone is frozen and used only as a feature extractor. A new `Linear(512, 8)` layer is trained for the eight RAVDESS emotions. The spectrogram remains `(1, 64, 301)` in the dataset and is copied to `(3, 64, 301)` inside the model.

In [ ]:
seed_everything(SEED)
model = EmotionResNet18(n_classes=N_CLASSES, pretrained=True).to(DEVICE)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print()
print(f"Total parameters    : {total_params:,}")
print(f"Frozen parameters   : {frozen_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable fraction  : {100 * trainable_params / total_params:.4f}%")

# Sanity check: the dataset provides one channel; forward() copies it to 3 channels.
with torch.no_grad():
    dummy = torch.zeros(1, *INPUT_SHAPE, device=DEVICE)
    logits = model(dummy)
    print()
    print(f"Dataset input       : {tuple(dummy.shape)}")
    print(f"ResNet sees         : (1, 3, {EXPECTED_SHAPE[0]}, {EXPECTED_SHAPE[1]})")
    print(f"Output logits       : {tuple(logits.shape)}")

# Strict check for the course requirement: only the new head may be trainable.
trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
print()
print("Trainable tensors:", trainable_names)
assert trainable_names == ["classifier.weight", "classifier.bias"]


## 9. Loss, optimiser, scheduler and metrics

In [ ]:
train_counts = np.bincount(y_all[idx["train"]], minlength=N_CLASSES)
if HPARAMS["class_weighted_loss"]:
    class_weights = torch.tensor(train_counts.sum() / (N_CLASSES * train_counts), dtype=torch.float32)
else:
    class_weights = torch.ones(N_CLASSES)
print("Class weights:", {c: round(float(w), 3) for c, w in zip(EMOTIONS, class_weights)})

criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_parameters, lr=HPARAMS["learning_rate"],
                              weight_decay=HPARAMS["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5,
                                                       patience=4, min_lr=1e-5)


def classification_metrics(y_true, y_pred):
    labels = list(range(N_CLASSES))
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "uar": recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
    }


def run_epoch(model, loader, train=False, augment_gen=None):
    # One pass over a loader. Returns loss, metrics and the predictions.
    model.train(train)
    total_loss, n_seen = 0.0, 0
    all_true, all_pred, all_prob, all_idx = [], [], [], []
    with torch.set_grad_enabled(train):
        for xb, yb, ib in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if train and augment_gen is not None:
                xb = augment_batch(xb, augment_gen)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * yb.size(0)
            n_seen += yb.size(0)
            probs = torch.softmax(logits.detach(), dim=1)
            all_true.append(yb.cpu().numpy())
            all_pred.append(probs.argmax(1).cpu().numpy())
            all_prob.append(probs.cpu().numpy())
            all_idx.append(ib.numpy())
    y_true, y_pred = np.concatenate(all_true), np.concatenate(all_pred)
    return {"loss": total_loss / n_seen, **classification_metrics(y_true, y_pred),
            "y_true": y_true, "y_pred": y_pred, "probs": np.concatenate(all_prob), "index": np.concatenate(all_idx)}


print("Training utilities defined.")

## 10. Training

Each epoch: one augmented pass over the training speakers, then one clean pass over the validation speakers.
The checkpoint with the best validation UAR (ties broken by lower validation loss) is saved to `best_model.pt`.
Training stops after 12 epochs without improvement. Test speakers are not touched here.

In [ ]:
history = []
best = {"uar": -1.0, "loss": float("inf"), "epoch": 0}
epochs_without_improvement = 0
train_start = time.time()

for epoch in range(1, HPARAMS["max_epochs"] + 1):
    t_epoch = time.time()
    train_res = run_epoch(model, loaders["train"], train=True, augment_gen=augment_generator)
    val_res = run_epoch(model, loaders["validation"], train=False)
    lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_res["uar"])

    history.append({
        "epoch": epoch, "lr": lr,
        "train_loss": train_res["loss"], "train_accuracy": train_res["accuracy"], "train_uar": train_res["uar"],
        "val_loss": val_res["loss"], "val_accuracy": val_res["accuracy"],
        "val_uar": val_res["uar"], "val_macro_f1": val_res["macro_f1"],
    })

    improved = (val_res["uar"] > best["uar"] + 1e-6 or
                (abs(val_res["uar"] - best["uar"]) <= 1e-6 and val_res["loss"] < best["loss"]))
    if improved:
        best = {"uar": val_res["uar"], "loss": val_res["loss"], "epoch": epoch,
                "accuracy": val_res["accuracy"], "macro_f1": val_res["macro_f1"]}
        torch.save({
            "state_dict": model.state_dict(),
            "model_config": model.config,
            "classes": EMOTIONS,
            "input_shape": list(INPUT_SHAPE),
            "epoch": epoch,
            "val_uar": float(val_res["uar"]),
            "val_loss": float(val_res["loss"]),
        }, BEST_MODEL_PATH)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f"Epoch {epoch:>3} | lr {lr:.1e} | train loss {train_res['loss']:.3f} acc {train_res['accuracy']:.3f} | "
          f"val loss {val_res['loss']:.3f} acc {val_res['accuracy']:.3f} UAR {val_res['uar']:.3f} "
          f"F1 {val_res['macro_f1']:.3f} | {time.time() - t_epoch:.1f}s{'  <- best' if improved else ''}")

    if epochs_without_improvement >= HPARAMS["early_stopping_patience"]:
        print(f"\nEarly stopping: no validation-UAR improvement for {epochs_without_improvement} epochs.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(MODEL_DIR / "training_history.csv", index=False)
print(f"\nTraining time: {(time.time() - train_start) / 60:.1f} min | best epoch {best['epoch']} | "
      f"best validation UAR {best['uar']:.3f}")

## 11. Learning curves

The dotted line marks the selected epoch. Training metrics are measured on augmented batches with dropout active,
so they are not directly comparable to validation values; the trend matters more than the gap.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), layout="constrained")
e = history_df["epoch"]

axes[0].plot(e, history_df["train_loss"], label="training")
axes[0].plot(e, history_df["val_loss"], label="validation")
axes[0].set_title("Loss (class-weighted cross-entropy)")

axes[1].plot(e, history_df["train_accuracy"], label="training")
axes[1].plot(e, history_df["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")

axes[2].plot(e, history_df["val_uar"], label="validation UAR", color="C2")
axes[2].plot(e, history_df["val_macro_f1"], label="validation macro F1", color="C3", linestyle="--")
axes[2].axhline(1 / N_CLASSES, color="grey", linewidth=0.8, label=f"chance ({1 / N_CLASSES:.3f})")
axes[2].set_title("Validation UAR and macro F1")

for ax in axes:
    ax.axvline(best["epoch"], color="black", linestyle=":", linewidth=1)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()
fig.savefig(MODEL_DIR / "training_curves.png", dpi=200)
plt.show()

## 12. Final evaluation on the unseen test speakers

The best checkpoint is reloaded from disk into a fresh model and evaluated **once** on the test speakers.
No decision (epochs, hyperparameters, checkpoint) was based on these speakers.

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True)
best_model = EmotionResNet18(**checkpoint["model_config"], pretrained=False).to(DEVICE)
best_model.load_state_dict(checkpoint["state_dict"])
best_model.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']} (validation UAR {checkpoint['val_uar']:.3f})\n")

test_res = run_epoch(best_model, loaders["test"], train=False)
y_true_test, y_pred_test = test_res["y_true"], test_res["y_pred"]

print(f"Test speakers : {', '.join(test_speakers)} ({len(y_true_test)} recordings)")
print(f"Test loss     : {test_res['loss']:.4f}")
print(f"Test accuracy : {test_res['accuracy']:.4f}")
print(f"Test UAR      : {test_res['uar']:.4f}   (macro recall)")
print(f"Test macro F1 : {test_res['macro_f1']:.4f}")

precision, recall, f1, support = precision_recall_fscore_support(
    y_true_test, y_pred_test, labels=list(range(N_CLASSES)), zero_division=0)
per_class = pd.DataFrame({"precision": precision, "recall": recall, "f1": f1, "support": support},
                         index=EMOTIONS).round(3)
print("\nPer-class metrics:")
display(per_class)

print("Classification report:")
print(classification_report(y_true_test, y_pred_test, labels=list(range(N_CLASSES)),
                            target_names=EMOTIONS, digits=3, zero_division=0))

test_predictions = data.loc[test_res["index"], ["filename", "speaker_key", "sex", "emotion", "intensity", "wav_path"]].copy()
test_predictions["predicted"] = [INDEX_TO_CLASS[i] for i in y_pred_test]
test_predictions["confidence"] = test_res["probs"].max(axis=1)
test_predictions["correct"] = test_predictions["emotion"] == test_predictions["predicted"]
for k, emotion in enumerate(EMOTIONS):
    test_predictions[f"p_{emotion}"] = test_res["probs"][:, k]
test_predictions.to_csv(MODEL_DIR / "test_predictions.csv", index=False)

## 13. Confusion matrix

Rows are normalised: each row shows how the clips of one **true** emotion were distributed over the predictions,
so the diagonal is the per-class recall.

In [ ]:
cm_counts = confusion_matrix(y_true_test, y_pred_test, labels=list(range(N_CLASSES)))
row_totals = cm_counts.sum(axis=1, keepdims=True)
cm = np.divide(cm_counts, row_totals, out=np.zeros_like(cm_counts, dtype=float), where=row_totals > 0)

fig, ax = plt.subplots(figsize=(8.5, 7.2), layout="constrained")
image = ax.imshow(cm * 100, cmap="Blues", vmin=0, vmax=100)
ax.set_xticks(range(N_CLASSES), EMOTIONS, rotation=45, ha="right")
ax.set_yticks(range(N_CLASSES), EMOTIONS)
ax.set_xlabel("Predicted emotion")
ax.set_ylabel("True emotion")
ax.set_title(f"Model 1 (CNN from scratch), unseen test speakers\n"
             f"row-normalised, %  |  UAR {test_res['uar']:.3f}, accuracy {test_res['accuracy']:.3f}")
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, f"{cm[i, j] * 100:.0f}", ha="center", va="center", fontsize=10,
                color="white" if cm[i, j] > 0.55 else "black")
fig.colorbar(image, ax=ax, label="% of true-class clips")
fig.savefig(MODEL_DIR / "confusion_matrix_test.png", dpi=200)
plt.show()

confusions = sorted(((cm[i, j], i, j) for i in range(N_CLASSES) for j in range(N_CLASSES)
                     if i != j and cm_counts[i, j] > 0), reverse=True)
print("Largest confusions:")
for share, i, j in confusions[:6]:
    print(f"  {share * 100:4.0f}% of {EMOTIONS[i]} samples were predicted as {EMOTIONS[j]} "
          f"({cm_counts[i, j]}/{row_totals[i, 0]})")

### Error analysis: per speaker, per intensity, confidence

Pooled test scores hide how much the model depends on **who** is speaking. UAR is averaged only over the emotions
present in each group (strong intensity has no neutral clips, so it has 7 classes). These tables are saved as
`test_per_speaker.csv` and `test_per_intensity.csv`.

In [ ]:
analysis_cols = ["emotion", "predicted", "correct", "confidence"]


def group_scores(g):
    present = sorted(g["emotion"].unique())          # average only over classes present in the group
    return pd.Series({
        "clips": len(g),
        "classes": len(present),
        "accuracy": g["correct"].mean(),
        "UAR": recall_score(g["emotion"], g["predicted"], labels=present, average="macro", zero_division=0),
        "mean confidence": g["confidence"].mean(),
    })


per_speaker = (test_predictions.groupby("speaker_key")[analysis_cols].apply(group_scores)
               .astype({"clips": int, "classes": int}))
per_intensity = (test_predictions.groupby("intensity")[analysis_cols].apply(group_scores)
                 .astype({"clips": int, "classes": int}))
confidence_summary = test_predictions.groupby("correct")["confidence"].describe()

print("Per test speaker:")
display(per_speaker.round(3))
print(f"Per-speaker UAR: mean {per_speaker['UAR'].mean():.3f} +/- {per_speaker['UAR'].std(ddof=1):.3f} (std over "
      f"{len(per_speaker)} test speakers)")
print("\nPer intensity:")
display(per_intensity.round(3))
print("Confidence, correct vs incorrect:")
display(confidence_summary.round(3))

per_speaker.to_csv(MODEL_DIR / "test_per_speaker.csv")
per_intensity.to_csv(MODEL_DIR / "test_per_intensity.csv")

## 14. Inspect test predictions

Randomly chosen (fixed seed) correct and incorrect test predictions, shown on the **raw dB** log-Mel matrix,
exactly as the web page will display it. Titles in green are correct, in red incorrect.

In [ ]:
inspect_rng = np.random.default_rng(SEED)
correct_rows = test_predictions[test_predictions["correct"]]
wrong_rows = test_predictions[~test_predictions["correct"]]
picked = pd.concat([
    correct_rows.iloc[inspect_rng.permutation(len(correct_rows))[:3]],
    wrong_rows.iloc[inspect_rng.permutation(len(wrong_rows))[:3]],
])
display(picked[["filename", "speaker_key", "emotion", "predicted", "confidence", "correct"]].round(3))

n_show = len(picked)
if n_show:
    n_cols = 3
    n_rows = int(np.ceil(n_show / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.6 * n_rows), layout="constrained", squeeze=False)
    for ax, (row_index, row) in zip(axes.ravel(), picked.iterrows()):
        title = (f"true {row['emotion']} | pred {row['predicted']} ({row['confidence'] * 100:.0f}%)\n"
                 f"{row['speaker_key']} | {row['filename']}")
        plot_spectrogram(X_db[row_index], ax, title, vmin=DB_FLOOR, vmax=float(X_db[picked.index].max()))
        ax.title.set_color("green" if row["correct"] else "red")
    for ax in axes.ravel()[n_show:]:
        ax.set_visible(False)
    plt.show()

for _, row in wrong_rows.iloc[inspect_rng.permutation(len(wrong_rows))[:2]].iterrows():
    print(f"Misclassified: {row['filename']} | true {row['emotion']} | predicted {row['predicted']}")
    display(Audio(filename=row["wav_path"]))

## 15. Export everything the web application needs

`/kaggle/working/model2/` becomes a self-contained package. The last check reloads it **from the saved files only** and runs the complete web chain (audio file → log-Mel → normalisation → frozen ResNet-18 + trained head) on a test recording, confirming that it reproduces the notebook's prediction.

In [ ]:
test_metrics = {
    "model": "Model 2: ResNet-18 transfer learning (frozen backbone)",
    "best_epoch": int(checkpoint["epoch"]),
    "validation": {k: float(v) for k, v in best.items() if k != "epoch"},
    "test": {
        "speakers": test_speakers, "n_recordings": int(len(y_true_test)),
        "loss": float(test_res["loss"]), "accuracy": float(test_res["accuracy"]),
        "uar": float(test_res["uar"]), "macro_f1": float(test_res["macro_f1"]),
        "per_class": per_class.reset_index(names="emotion").to_dict(orient="records"),
        "confusion_matrix_counts": cm_counts.tolist(),
        "confusion_matrix_row_normalised": np.round(cm, 4).tolist(),
        "per_speaker": per_speaker.round(4).reset_index().to_dict(orient="records"),
        "per_intensity": per_intensity.round(4).reset_index().to_dict(orient="records"),
    },
}
artifacts = {
    "class_to_index.json": CLASS_TO_INDEX,
    "index_to_class.json": {str(k): v for k, v in INDEX_TO_CLASS.items()},
    "preprocessing_config.json": PREPROCESSING_CONFIG,
    "model_config.json": {**best_model.config,
                          "stored_input_shape": list(INPUT_SHAPE),
                          "resnet_input_shape": list(RESNET_INPUT_SHAPE),
                          "architecture": "ImageNet ResNet-18 frozen backbone -> Linear(512, n_classes)",
                          "channel_adaptation": "repeat the single spectrogram channel 3 times",
                          "backbone": "ResNet-18",
                          "pretraining": "ImageNet",
                          "backbone_frozen": True,
                          "total_parameters": total_params,
                          "trainable_parameters": trainable_params},
    "training_config.json": {**HPARAMS, "seed": SEED, "device": str(DEVICE),
                             "optimizer": "AdamW (trainable head only)", "loss": "cross-entropy",
                             "class_weights": class_weights.tolist()},
    "test_metrics.json": test_metrics,
}
for name, content in artifacts.items():
    (MODEL_DIR / name).write_text(json.dumps(content, indent=2))

# --- Reload from files only and run the full web-app chain ----------------------
web_model, web_normalizer, web_classes = ser_model2.load_model(MODEL_DIR, device="cpu")
sample = test_predictions.iloc[0]
result = ser_model2.predict(sample["wav_path"], web_model, web_normalizer, web_classes)
notebook_probs = sample[[f"p_{e}" for e in EMOTIONS]].to_numpy(dtype=float)
web_probs = np.array([result["probabilities"][e] for e in EMOTIONS])
print(f"Web-chain check on {sample['filename']}: predicted {result['emotion']} ({result['confidence']:.3f}); "
      f"notebook predicted {sample['predicted']} ({sample['confidence']:.3f}); "
      f"max probability difference {np.abs(web_probs - notebook_probs).max():.1e}")
assert result["emotion"] == sample["predicted"] and np.allclose(web_probs, notebook_probs, atol=1e-4)

print()
print(f"Saved in {MODEL_DIR}:")
for path in sorted(MODEL_DIR.iterdir()):
    if path.is_file():
        print(f"  {path.name:<32} {path.stat().st_size / 1024:>9.1f} KiB")


## 16. Final summary

In [ ]:
def fmt(ids):
    return ", ".join(ids)

print("MODEL 2: RESNET-18 TRANSFER LEARNING (FROZEN BACKBONE)")
print()
print(f"Training speakers   : {fmt(train_speakers)}  ({len(idx['train'])} recordings)")
print(f"Validation speakers : {fmt(val_speakers)}  ({len(idx['validation'])} recordings)")
print(f"Test speakers       : {fmt(test_speakers)}  ({len(idx['test'])} recordings)")
print()
print(f"Number of classes   : {N_CLASSES}")
print(f"Stored input shape  : {INPUT_SHAPE}")
print(f"ResNet input shape  : {RESNET_INPUT_SHAPE}")
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} (classification head only)")
print()
print(f"Best epoch          : {best['epoch']}")
print(f"Best validation UAR : {best['uar']:.3f}")
print()
print(f"Test accuracy       : {test_res['accuracy']:.3f}")
print(f"Test UAR            : {test_res['uar']:.3f}")
print(f"Test macro F1       : {test_res['macro_f1']:.3f}")
print(f"Per-speaker test UAR: {per_speaker['UAR'].mean():.3f} +/- {per_speaker['UAR'].std(ddof=1):.3f} "
      f"(range {per_speaker['UAR'].min():.3f} to {per_speaker['UAR'].max():.3f})")
print()
print(f"Best model saved to : {BEST_MODEL_PATH}")
print(f"Web-app package     : {MODEL_DIR} (ser_model2.py + weights + normalization + configs)")
